# Пример bootstrap на синтетических данных

Учебный пример к **занятию 5**: оценка неопределённости модели методом bootstrap.

Работаем с синтетическими данными — целевая переменная $y$ есть многочлен 3-й степени от $x$ плюс гауссов шум. «Правильные» коэффициенты нам известны, поэтому можно проверить, насколько хорошо bootstrap восстанавливает и коэффициенты модели, и качество её прогноза.

План работы:

1. породить данные с известными коэффициентами;
2. собрать модель полиномиальной регрессии из стандартных блоков `scikit-learn`;
3. обсудить, из чего складывается неопределённость прогноза (алеаторическая и эпистемическая);
4. реализовать bootstrap своими руками и получить распределения оценок коэффициентов и метрики RMSE.

### Полезные ссылки
- <a href="https://numpy.org/">Документация NumPy</a>
- <a href="https://pandas.pydata.org/">Документация pandas</a>
- <a href="https://matplotlib.org/">Документация Matplotlib</a>
- <a href="https://scikit-learn.org/stable/">Документация scikit-learn</a>
- <a href="https://en.wikipedia.org/wiki/Bootstrapping_(statistics)">Bootstrap (сэмплирование с возвращением)</a>
- <a href="https://tqdm.github.io/">tqdm — индикатор прогресса</a>

### Дополнительные материалы
- Ноутбук `Lect05a_CLT_demo.ipynb` — выборочные статистики и центральная предельная теорема
- Ноутбук `Lect05c_CV_real_data.ipynb` — кросс-валидация на реальных данных
- [sklearn.preprocessing.PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
- [sklearn.metrics.mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)


## Импорт библиотек

Подключаем числовые и табличные инструменты, графику, индикатор прогресса `tqdm` и нужные блоки `scikit-learn`: линейную регрессию, порождение полиномиальных признаков, масштабирование данных и метрики качества.

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [tqdm](https://tqdm.github.io/), [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics), [sklearn.preprocessing](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.preprocessing), [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

%matplotlib inline 
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.dpi'] = 150

## Генерация данных 

Порождаем синтетические данные с зависимость y от x в форме многочлена 3-й степени. Для этого используем модель препроцессинга [PolynomialFeatures](https://scikit-learn.org/dev/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) библиотеки sklearn.

`true_degee` и `true_theta` задают «истинную» зависимость, `true_noise` — стандартное отклонение гауссова шума. На графике видно, что точки ложатся вдоль кубической параболы, но с заметным разбросом.

**Инструменты:** [numpy.linspace](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html), [numpy.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html), [numpy.dot](https://numpy.org/doc/stable/reference/generated/numpy.dot.html), [matplotlib.pyplot.plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html).


In [ ]:

true_degee = 3; true_theta = [-50, -4, 1.9, 0.25]
#true_degee = 4; true_theta = [0, -2.5, -1.9, -0.019, 0.021]

true_noise = 10


def generate_data (sample_size = 40):

    # x — равномерная сетка с небольшим случайным разбросом
    X = np.linspace(-10, 10, sample_size) + np.random.randn(sample_size)*0.25
    X = X.reshape(-1,1)

    # полиномиальные признаки: 1, x, x^2, x^3
    poly = PolynomialFeatures(true_degee)
    X_poly = poly.fit_transform(X.reshape(-1, 1))
    print('X shape: ', X.shape)
    print('X_poly shape: ', X_poly.shape)

    display(pd.DataFrame (X).head())
    display(pd.DataFrame (X_poly).head())

    # y = true_theta * X_poly + гауссов шум
    Y = np.dot(X_poly, true_theta) + np.random.randn(sample_size) * true_noise  #+ np.random.randn(sample_size) * np.sqrt (np.abs(X)) * 2
    Y = Y.reshape(-1,1)
    return X, Y

X, Y = generate_data (10000)

plt.figure()
plt.plot(X, Y, '.k')
plt.xlabel ('Признак X')
plt.ylabel ('Целевая переменная Y')
plt.title ('Синтетические данные: многочлен 3-й степени + гауссов шум')
plt.ylim([-100, 100])
plt.xlim([-10, 10])
plt.grid (alpha=0.3)


## Создаем новый класс моделей машинного обучения - полиномиальную регрессию

Данная модель объединяет модель линейной регресии [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html), модель порождения полиномиальных признаков   [PolynomialFeatures](https://scikit-learn.org/dev/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) и, опционально, произвольную модель масштабирования данных.  

Модель имеет два **гиперпараметра**:
- Степень полинома
- Модель масштабирования данных (по умолчанию отсутствует)

Обратите внимание на две детали реализации:

- `LinearRegression(fit_intercept=False)` — свободный член не оценивается, потому что константный признак уже добавлен внутри `PolynomialFeatures(include_bias=True)`;
- в `fit` масштабирование обучается (`fit_transform`), а в `predict` только применяется (`transform`): иначе на новых данных получалось бы другое преобразование.

**Инструменты:** [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [требования к объектам с интерфейсом fit/predict](https://scikit-learn.org/stable/developers/develop.html).


In [ ]:
class PolynomialRegression:
    def __init__(self, degree = 1, scaler = None):
        self.degree = degree
        self.regressor = LinearRegression(fit_intercept=False)
        self.scaler = scaler () if scaler is not None else None
        
    def fit(self, X, Y):
        self.poly = PolynomialFeatures(self.degree, include_bias = True)
        X_poly = self.poly.fit_transform(X.reshape(-1, 1))
        if self.scaler is not None:
            X_poly = self.scaler.fit_transform(X_poly)
        self.regressor.fit(X_poly, Y)
        
    def predict(self, X):
        X_poly = self.poly.fit_transform(X.reshape(-1, 1))
        if self.scaler is not None:
            X_poly = self.scaler.transform(X_poly)       
        return self.regressor.predict(X_poly)

# Виды неопределенности аппроксимированой целевой переменной
### Алеаторическая неопределенность
Алеаторическая - неопределенность, наследуемая из неопределенности в данных. В рассматриваемой модели линейной регрессии алеаторической неопределенности соответствует ошибка, наследуемая от шума $\epsilon \sim \mathcal{N}(0, \sigma^2)$<br />
Алеаторическую неопределенность **невозможно снизить**. Это базовое свойство той закономерности, которая порождает данные. Можно рассматривать другую закономерность (другую постановку задачи, другой дизайн эксперимента). Но это будет уже другая задача, другая история.

### Эпистемическая неопределенность
Эпистемическая - неопределенность, обусловленная недостаточностью знаний, сведений, способностей модели, *etc*. В рассматриваемой модели линейной регрессии эпистемической соответствует неопределенность, обусловленная чувствительностью модели к выбросам, недостаточной выразительностью модели, недостаточной обобщающей способностью модели, плохим балансом между обобщающей способностью и выразительностью модели, малым количеством данных.
Эпистемическую неопределенность можно и нужно снижать. Среди путей снижения эпистемической неопределенности в задаче регрессии:

- Увеличение выборки (data mining);
- Фильтрация выбросов в данных (предобработка данных);
- Отбор вида модели: выбор модели, менее чувствительной к выбросам в данных, обладающей лучшей обобщающей способностью, обладающей лучшей выразительностью; правильный подбор гиперпараметров, отвечающих за баланс между обобщающей способностью и выразительностью модели;

### Помимо оценки качества "финальной" модели обязательно нужно оценивать эпистемическую неопределенность
Именно эта неопределенность характеризует, насколько хорошо подходит эта модель для описания имеющихся данных.

Ошибка (качество) модели - оценивается на тестовой выборке. В оценке ошибки модели заложены оба типа неопределенности. Оценка качества модели по какой-то мере качества (например, MSE) - это оценка ошибки **одного** варианта модели, оптимизированного на **конкретной** тренировочной выборке.

**Идеальная модель** имеет ошибку, совпадающую с алеаторичской неопределенностью (шумом в данных, $\epsilon$).

<b>НО!!!</b><br />
В настоящий момент НЕТ простого способа разделить алеаторическую и эпистемическую неопределенности.<br />
Способы разделения этих неопределенностей - открытая область исследований. Наиболее продвинутые результаты получены с применением байесовских подходов (не рассматриваются в этом курсе)


## Bootstrap (бутстрэп, бутстреп)

**Bootstrap** - метод сэмплирования из выборки, предоставляющий возможность оценивать статистики на сокращенной выборке.

**Bootstrap** - метод сэмплирования с возвратом:

1. Определиться с размером bootstrap-выборки $N_b$
2. Повторять до тех пор, пока размер bootstrap-выборки не достигнет $N_b$:
    - Выбрать произвольный элемент исходной выборки, "вернуть" его на место.
    - Добавить выбранный элемент в bootstrap-выборку.


## Оценка качества модели с применением подхода Bootstrap
1. Определиться с размером bootstrap-выборки $N_b$
2. Определиться с количеством bootstrap-выборок $B$
3. Для каждой bootstrap-выборки:<br />
    a. Получить bootstrap-выборку. Оставшиеся события - т.н. out-of-bag выборка (OOB)<br />
    b. Оптимизировать модель на этой выборке как на тренировочной<br />
    c. Оценить качество модели по выбранной мере качества на OOB-выборке.<br />
4. Оценить качество модели как среднее оценок, получаемых в п.**3.c.**

Получаемая таким образом оценка ошибки включает алеаторическую и эпистемическую неопределенности.

## Оценка значений и неопределенностей параметров модели с применением подхода Bootstrap
1. Определиться с размером bootstrap-выборки $N_b$
2. Определиться с количеством bootstrap-выборок $B$
3. Для каждой bootstrap-выборки:<br />
    a. Получить bootstrap-выборку.<br />
    b. Оптимизировать модель на этой выборке как на тренировочной, записать полученные параметры.<br />
4. Оценить значения параметров модели как выборочную оценку среднего для параметров, получаемых в п.**3.b.**
5. Оценить неопределенность параметров модели как выборочную оценку стандартного отклонения параметров, получаемых в п.**3.b.**

## Оценка неопределенности вывода модели с применением подхода Bootstrap
1. Определиться с размером bootstrap-выборки $N_b$
2. Определиться с количеством bootstrap-выборок $B$
3. Для каждой bootstrap-выборки:<br />
    a. Получить bootstrap-выборку<br />
    b. Оптимизировать модель на этой выборке как на тренировочной. Записать значения параметров.<br />
4. На новых элементах, к которым следует применить модель, оценивать эпистемическую неопределенность модели следующим образом:
    a. Применить модель в новому объекту с каждым из наборов параметров модели, полученных на этапе **3.b**. ВНИМАНИЕ! Эти значения - аппроксимация целевой переменной **без учета шума $\epsilon$**. То есть, это оценка среднего значения $y_i = \theta\bar{x_i}$ <br/>
    b. Если есть обоснованные предположения относительно параметров распределения шума $\epsilon$ - к каждому значению полученного среднего следует добавить шум.
    b. Оценить неопределенность полученных значений как выборочную оценку стандартного отклонения.

## Программная реализация Bootstrap своими руками

Функция `sample_bootstrap` берёт `sample_size` индексов от 0 до `X.shape[0] - 1` **случайно и с возвращением** (`np.random.randint`) — это и есть bootstrap-выборка. Наблюдения, которые в неё не попали, образуют out-of-bag выборку (OOB): на них удобно оценивать качество модели, ведь в обучении они не участвовали.

Флаг `return_OOB_sample` переключает режим работы функции: вернуть только обучающую часть или обучающую и OOB вместе.

**Инструменты:** [numpy.random.randint](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randint.html), [numpy.arange](https://numpy.org/doc/stable/reference/generated/numpy.arange.html).


In [ ]:
def sample_bootstrap(X, Y, sample_size, return_OOB_sample):
    # индексы берутся случайно и с возвращением — это bootstrap-выборка
    sample_indices = np.random.randint(0,X.shape[0], sample_size)
    if return_OOB_sample:
        # всё, что не попало в bootstrap-выборку, — out-of-bag (OOB)
        oob_indices = np.array([i for i in np.arange(0, X.shape[0], 1, dtype=int) if i not in sample_indices])
        return X[sample_indices, :], Y[sample_indices, :], X[oob_indices, :], Y[oob_indices, :]
    else:
        return X[sample_indices, :], Y[sample_indices, :]


## Оценка неопределенности параметров модели и метрик качества

Запускаем bootstrap `B = 1000` раз. На каждой итерации:

1. получаем bootstrap-выборку размера `sample_size = 100` и OOB-выборку;
2. обучаем полиномиальную регрессию 3-й степени на bootstrap-выборке;
3. считаем RMSE на OOB-выборке и запоминаем коэффициенты модели.

В результате получаем `model_theta` — оценки коэффициентов (по одной строке на каждую bootstrap-выборку) и `model_quality` — набор из `B` оценок RMSE. Индикатор прогресса `tqdm` показывает, что расчёт идёт.

**Инструменты:** [tqdm](https://tqdm.github.io/), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [numpy.sqrt](https://numpy.org/doc/stable/reference/generated/numpy.sqrt.html), [numpy.copy](https://numpy.org/doc/stable/reference/generated/numpy.copy.html).


In [ ]:
B = 1000
sample_size = 100
model_theta = []
model_quality = []

for i in tqdm(range(B), total=B):
    # обучающая (bootstrap) и out-of-bag выборки
    X_train, Y_train, X_test, Y_test = sample_bootstrap(X,Y, sample_size=sample_size, return_OOB_sample=True)
    model = PolynomialRegression(degree=3) #, scaler=StandardScaler)
    model.fit(X_train,Y_train)
    Y_test_predicted = model.predict(X_test)
    # качество — RMSE на out-of-bag выборке, коэффициенты — на bootstrap-выборке
    model_quality.append(np.sqrt (mean_squared_error(Y_test, Y_test_predicted)))
    model_theta.append(np.copy(model.regressor.coef_))

model_theta = np.array(model_theta).squeeze()


### Полученные коэффициенты

Выводим массив `model_theta`: строки — bootstrap-выборки, столбцы — коэффициенты при $1, x, x^2, x^3$. Оценки «гуляют» вокруг истинных значений `true_theta`, и именно этот разброс далее используется как мера неопределённости параметров модели.

**Инструменты:** отображение массива [numpy.ndarray](https://numpy.org/doc/stable/reference/arrays.ndarray.html) средствами Jupyter.


In [ ]:
model_theta

### Прогноз на out-of-bag выборке

Сравниваем предсказанные и наблюдённые значения: идеальный прогноз лежал бы на диагонали.

Важная деталь: `Y_test` и `Y_test_predicted` — это результат **последней** итерации цикла bootstrap, а не всех сразу; они просто остались в памяти после его завершения. Для честной оценки качества смотрите на распределение RMSE в следующей ячейке.

**Инструменты:** [matplotlib.pyplot.plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html).


In [ ]:
plt.figure()
plt.plot(Y_test, Y_test_predicted, '.k')
plt.xlabel ('Наблюдённое значение Y (OOB-выборка)')
plt.ylabel ('Предсказание модели Y_predicted')
plt.title ('Bootstrap: прогноз на out-of-bag выборке (последняя итерация)')
plt.grid (alpha=0.3)


### Распределения оценок коэффициентов

Гистограммы показывают, как распределены оценки каждого коэффициента по всем bootstrap-выборкам. Розовая вертикальная линия — среднее по bootstrap-оценкам, зелёная — истинное значение `true_theta`; в заголовке панели указаны среднее и стандартное отклонение. Чем уже гистограмма, тем точнее оценивается коэффициент.

**Инструменты:** [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html), [matplotlib.pyplot.hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html), [matplotlib.pyplot.vlines](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.vlines.html), [matplotlib.lines.Line2D](https://matplotlib.org/stable/api/_as_gen/matplotlib.lines.Line2D.html) (для элементов легенды).


In [ ]:
from matplotlib.lines import Line2D

f = plt.figure(figsize=(10,4), dpi=300)
for i in range(model_theta.shape[1]):
    plt.subplot(1,4, i+1)
    h_counts,h_bins,_ = plt.hist(model_theta[:,i], bins=30, color='white', ec="black", linewidth=0.5)
    mean_theta_i = np.mean(model_theta[:,i])
    std_theta_i = np.std(model_theta[:,i])
    y_lim = plt.ylim()

    plt.vlines(mean_theta_i,  y_lim[0], y_lim[1], color='magenta')
    plt.vlines(true_theta[i], y_lim[0], y_lim[1], color='green')
    plt.ylim(y_lim)
    plt.title("$\\theta_%d$ \nmean = %.3f \n std = %.3f" % (i, mean_theta_i, std_theta_i))
    plt.xlabel ('Значение коэффициента')
    if i == 0:
        plt.ylabel ('Число наблюдений')
        plt.legend (handles=[Line2D([], [], color='magenta', label='среднее bootstrap-оценок'),
                             Line2D([], [], color='green', label='истинное значение')],
                    fontsize=7, loc='upper right')

_ = plt.tight_layout()


### Распределение метрики качества RMSE

Гистограмма показывает разброс RMSE по всем bootstrap-выборкам: это и есть оценка неопределённости качества модели, а не единственное «число качества». В заголовке — среднее и стандартное отклонение RMSE. Сравните ширину этой гистограммы со «разбросом» модели, обученной один раз на одной выборке.

**Инструменты:** [matplotlib.pyplot.hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html).


In [ ]:
h_counts,h_bins,_ = plt.hist(model_quality, bins=30, color='white', ec="black", linewidth=0.5)
plt.xlabel ('RMSE (на out-of-bag выборке)')
plt.ylabel ('Число bootstrap-выборок')
plt.title("RMSE \nmean = %.3f \n std = %.3f" % (np.mean(model_quality), np.std(model_quality)))
